In [2]:
from azureml.core import Workspace, Dataset, Datastore

subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'

ws = Workspace(subscription_id=subscription_id,
               resource_group=resource_group,
               workspace_name=workspace_name)


datastore = Datastore.get(ws, "researcher_data")


dataset = Dataset.Tabular.from_parquet_files(
    path=[(datastore, 'Zahra/032026/Data_OOT/MEDS_MDS/data/train/*.parquet')]    #     Zahra/Data-07-2025/MDP/MEDS_811/data/train
)

df = dataset.to_pandas_dataframe()
df.head(15)


Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


In [5]:
from azureml.core import Dataset
from azureml.core import Workspace, Dataset, Datastore

# اتصال به Workspace
subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'

ws = Workspace(subscription_id=subscription_id,
               resource_group=resource_group,
               workspace_name=workspace_name)

# گرفتن datastore
datastore = Datastore.get(ws, "researcher_data")


base_prefix = 'Zahra/Data-07-2025/MD/MEDS_811/data/train'

# همه‌ی پارکت‌های زیرشاخه‌ها:
file_ds = Dataset.File.from_files(path=[(datastore, f'{base_prefix}/*.parquet')])

# بدون دانلود، لیست مسیر فایل‌ها را بده:
paths = file_ds.to_path()  # لیست مسیرهای واقعی فایل‌ها

print('Parquet files:', len(paths))
for p in paths[:5]:
    print(p)  # چند نمونه برای چک کردن

{'infer_column_types': 'False', 'activity': 'to_path'}
{'infer_column_types': 'False', 'activity': 'to_path', 'activityApp': 'FileDataset'}
Parquet files: 36
/0.parquet
/1.parquet
/10.parquet
/11.parquet
/12.parquet


In [14]:
from azureml.core import Workspace, Dataset, Datastore

# اتصال به Workspace
subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'

ws = Workspace(subscription_id=subscription_id,
               resource_group=resource_group,
               workspace_name=workspace_name)

# گرفتن datastore
datastore = Datastore.get(ws, "researcher_data")

# ساختن dataset از parquet
dataset = Dataset.Tabular.from_parquet_files(
    path=[(datastore, 'Zahra/Data-07-2025/MD/MEDS_811/data/train/12.parquet')]
)

# تبدیل به pandas DataFrame
df1 = dataset.to_pandas_dataframe()
df1.head(15)


{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


,subject_id,time,code,numeric_value
0,17,NaT,GENDER//Kvinde,NaN
1,17,1941-09-05 00:00:00,DOB,NaN
2,17,2018-09-10 00:00:00,D/DM774,NaN
3,17,2018-09-10 00:00:00,D/DM216O,NaN
4,17,2018-09-10 00:00:00,D/DM201,NaN
5,17,2019-01-16 00:00:00,D/DK573,NaN
6,17,2019-01-16 00:00:00,D/DK590A,NaN
7,17,2019-01-16 00:00:00,D/DK552,NaN
8,17,2019-01-16 00:00:00,D/DD126C,NaN
9,17,2019-01-16 08:55:00,ADMISSION_ADT,NaN


In [10]:
print("Before:", len(df))
df2 = df[~df['code'].str.startswith("P/")]
print("After:", len(df))


Before: 10129224
After: 10129224


In [19]:
df1.shape

(10059266, 4)

In [15]:
# شرط: آیا code با "P/" شروع میشه؟
df["is_p"] = df["code"].str.startswith("P/")

# groupby روی subject_id و بررسی
only_p_patients = (
    df.groupby("subject_id")["is_p"]
      .all()   # همه رکوردهای اون بیمار P/ باشن
)

# انتخاب patientهایی که فقط P/ دارن
only_p_patients = only_p_patients[only_p_patients].index

print(f"تعداد بیماران فقط P/: {len(only_p_patients)}")
print("لیست subject_id ها:", only_p_patients.tolist())


تعداد بیماران فقط P/: 0
لیست subject_id ها: []


In [16]:
df["is_p"].head()

0    False
1    False
2    False
3     True
4    False
Name: is_p, dtype: bool

In [17]:
df['is_p'].value_counts()

False    10129224
True      2339704
Name: is_p, dtype: int64

In [2]:
from azureml.core import Workspace, Dataset, Datastore

# اتصال به Workspace
subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'

ws = Workspace(subscription_id=subscription_id,
               resource_group=resource_group,
               workspace_name=workspace_name)

# گرفتن datastore
datastore = Datastore.get(ws, "researcher_data")

# خواندن همه فایل‌های parquet در مسیر مشخص
dataset = Dataset.Tabular.from_parquet_files(
    path=[(datastore, 'Zahra/Data-07-2025/MDP/MEDS_811/data/train/*.parquet')]
)

# تبدیل به pandas DataFrame
df = dataset.to_pandas_dataframe()
df.head(15)


{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


,subject_id,time,code,numeric_value
0,51,NaT,GENDER//Kvinde,NaN
1,51,1991-04-09 00:00:00,DOB,NaN
2,51,2018-05-29 00:00:00,D/DJ039,NaN
3,51,2020-02-19 10:57:00,P/UXUD10,NaN
4,51,2020-02-20 00:00:00,D/DK802,NaN
5,51,2020-02-24 08:25:00,P/AAF20,NaN
6,51,2020-02-24 08:25:00,P/ZZ0150,NaN
7,110,NaT,GENDER//Mand,NaN
8,110,1953-01-26 00:00:00,DOB,NaN
9,110,2016-05-31 08:08:00,P/UXCD60,NaN


In [3]:
# استخراج prefix از code
df['prefix'] = df['code'].str.extract(r'^([^/]+)')

# لیست prefixهای هر بیمار
prefix_per_subject = df.groupby('subject_id')['prefix'].unique().reset_index()
prefix_per_subject.head()


,subject_id,prefix
0,2,"[GENDER, DOB, P, D, ADMISSION_ADT, DISCHARGE_A..."
1,3,"[GENDER, DOB, D, P, ADMISSION_ADT, DISCHARGE_A..."
2,4,"[GENDER, DOB, D, P]"
3,6,"[GENDER, DOB, P, D, ADMISSION_ADT, DISCHARGE_A..."
4,7,"[GENDER, DOB, D, ADMISSION_ADT, DISCHARGE_ADT,..."


In [33]:
import pandas as pd

# پیدا کردن سطرهایی که فقط codeهای نوع /P دارن
only_p = df[df['code'].str.startswith('P/', na=False)]

# بیماران با فقط /P کد
subject_ids_only_p = only_p['subject_id'].unique()

# حالا بیماران با codeهای غیر از /P
not_p = df[~df['code'].str.startswith('P/', na=False)]
subject_ids_with_non_p = set(not_p['subject_id'].unique())

# حذف بیمارانی که فقط /P دارن
only_p_ids_to_exclude = [sid for sid in subject_ids_only_p if sid not in subject_ids_with_non_p]

print("بیمارانی که فقط کد P دارن:", only_p_ids_to_exclude)


بیمارانی که فقط کد P دارن: []


In [34]:
df_filtered = df[~df['code'].str.startswith('P/', na=False)]


In [35]:
df_filtered

,subject_id,time,code,numeric_value,prefix
0,51,NaT,GENDER//Kvinde,NaN,GENDER
1,51,1991-04-09,DOB,NaN,DOB
2,51,2018-05-29,D/DJ039,NaN,D
4,51,2020-02-20,D/DK802,NaN,D
7,110,NaT,GENDER//Mand,NaN,GENDER
...,...,...,...,...,...
447890401,2218025,2019-09-16,D/DN300,NaN,D
447890403,2218025,2021-01-04,D/DM774,NaN,D
447890405,2218025,2021-05-12,D/DR238A,NaN,D
447890409,2218025,2022-02-17,D/DM202,NaN,D


In [36]:
df

,subject_id,time,code,numeric_value,prefix
0,51,NaT,GENDER//Kvinde,NaN,GENDER
1,51,1991-04-09 00:00:00,DOB,NaN,DOB
2,51,2018-05-29 00:00:00,D/DJ039,NaN,D
3,51,2020-02-19 10:57:00,P/UXUD10,NaN,P
4,51,2020-02-20 00:00:00,D/DK802,NaN,D
...,...,...,...,...,...
447890408,2218025,2021-05-12 09:26:00,P/ZZ0150,NaN,P
447890409,2218025,2022-02-17 00:00:00,D/DM202,NaN,D
447890410,2218025,2022-02-17 09:02:00,P/UXRG50,NaN,P
447890411,2218025,2022-02-17 09:09:00,P/ZZ0151,NaN,P


In [4]:
# بیمارهایی که فقط DOB دارند
only_dob = prefix_per_subject[prefix_per_subject['prefix'].apply(lambda x: set(x).issubset({'DOB'}))]

# بیمارهایی که فقط DOB و DOD دارند
only_dob_dod = prefix_per_subject[prefix_per_subject['prefix'].apply(lambda x: set(x).issubset({'DOB','DOD'}))]

print("فقط DOB:", only_dob['subject_id'].tolist())
print("فقط DOB+DOD:", only_dob_dod['subject_id'].tolist())


فقط DOB: []
فقط DOB+DOD: []


In [17]:
df_noP = df[~df['code'].str.startswith('P/')]


NameError: name 'df' is not defined

In [2]:
print(df.describe(include='all'))

/tmp/ipykernel_3162/115838729.py:1: FutureWarning: Treating datetime data as categorical rather than numeric in `.describe` is deprecated and will be removed in a future version of pandas. Specify `datetime_is_numeric=True` to silence this warning and adopt the future behavior now.
  print(df.describe(include='all'))


          subject_id                 time       code  numeric_value
count   4.478904e+08            446115991  447890413            0.0
unique           NaN             29358285      31039            NaN
top              NaN  2019-02-02 08:00:00  M/N02BE01            NaN
freq             NaN                19988   22260711            NaN
first            NaN  1841-01-01 00:00:00        NaN            NaN
last             NaN  2024-08-22 18:30:00        NaN            NaN
mean    1.110206e+06                  NaN        NaN            NaN
std     6.398670e+05                  NaN        NaN            NaN
min     2.000000e+00                  NaN        NaN            NaN
25%     5.587940e+05                  NaN        NaN            NaN
50%     1.109295e+06                  NaN        NaN            NaN
75%     1.664584e+06                  NaN        NaN            NaN
max     2.218029e+06                  NaN        NaN            NaN


In [2]:
print(df['code'].nunique(), 'Number of Unique Codes')

31039 Number of Unique Codes


In [ ]:

print(df['subject_id'].nunique(), 'Number of Patients')


In [4]:
# اگر کد از نوع str نیست، تبدیل می‌کنیم
df['code'] = df['code'].astype(str)

# استخراج prefix — چیزی قبل از اولین '/'
df['code_prefix'] = df['code'].str.extract(r'^([^/]+/+)')

# پر کردن مقادیر خالی با خود کد
df['code_prefix'] = df['code_prefix'].fillna(df['code'])

# شمارش تعداد هر prefix
prefix_counts = df['code_prefix'].value_counts()
print(prefix_counts)


M/               280865103
P/                84395421
ADMISSION_ADT     22073280
DISCHARGE_ADT     22073280
MOVE_ADT          22073280
D/                12718535
GENDER//           1774422
DOB                1774419
DOD                 142673
Name: code_prefix, dtype: int64
